# UPC RAG — Scraper Completo (Fase 2)

Descarga PDFs y guarda texto HTML de páginas relevantes.
Usa `requests` para páginas estáticas y **Playwright** para SPAs.

Salida en `/tmp/upc_documents/` para evitar bloqueos de iCloud.

## 1 · Instalación de dependencias

In [15]:
%pip install -q requests beautifulsoup4 tqdm pandas lxml ipywidgets openpyxl playwright

# Instala Chromium (solo hace falta una vez, tarda ~1 min)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'playwright', 'install', 'chromium'],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else "")
print(result.stderr[-300:] if result.stderr else "")
print("✓ Playwright instalado" if result.returncode == 0 else f"✗ error: {result.returncode}")

Note: you may need to restart the kernel to use updated packages.


✓ Playwright instalado


## 2 · Imports y configuración global

In [17]:
import re
import csv
import time
import random
import shutil
import hashlib
from collections import deque
from datetime import datetime
from urllib.parse import urlparse, urljoin, urldefrag
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import pandas as pd

# ── Semillas (corregidas respecto a Fase 1: URLs muertas removidas) ──────────
SEED_URLS = [
    # Pregrado / Ingeniería de Sistemas
    "https://pregrado.upc.edu.pe/facultad-de-ingenieria/ingenieria-de-sistemas-de-informacion/",
    "https://pregrado.upc.edu.pe/facultad-de-ingenieria/",
    # Becas y financiamiento
    "https://www.upc.edu.pe/admision/becas-y-financiamiento/becas-internas-alumnos/",
    "https://www.upc.edu.pe/admision/becas-y-financiamiento/",
    "https://www.upc.edu.pe/servicios/becas-creditos-y-cobranzas/",
    # Explora UPC — el portal de ayuda al alumno más rico (Fase 1)
    "https://explora.upc.edu.pe/",
    "https://explora.upc.edu.pe/ayuda-upc",
    "https://explora.upc.edu.pe/79839-grados-titulos-y-mencion",
    "https://explora.upc.edu.pe/79837-pagos",
    "https://explora.upc.edu.pe/79836-otros-servicios",
    "https://explora.upc.edu.pe/79202-emision-de-documentos",
    "https://explora.upc.edu.pe/79831-asistencias-y-notas",
    "https://explora.upc.edu.pe/79179-retiros-academicos",
    "https://explora.upc.edu.pe/cachimbos",
    "https://explora.upc.edu.pe/alumno-postgrado",
]

# ── Parámetros de crawl ───────────────────────────────────────────────────────
MAX_DEPTH     = 2
DELAY_MIN     = 2.0
DELAY_MAX     = 4.0
TIMEOUT       = 20       # segundos para requests
MAX_PAGES     = 600      # cap defensivo
SPA_THRESHOLD = 200      # palabras; si hay menos → fallback a Playwright

USER_AGENT = (
    "Mozilla/5.0 (compatible; UPC-RAG-Scraper/2.0; "
    "+https://github.com/upc-thesis/chatbot)"
)
ALLOWED_DOMAIN_SUFFIX = ".upc.edu.pe"

SKIP_PATTERNS = [
    "login", "signin", "campus-net", "intranet", "blackboard",
    "canvas", "aula-virtual", "facebook.com", "twitter.com", "x.com",
    "instagram", "youtube", "linkedin", "whatsapp",
]
SKIP_SCHEMES = ("mailto:", "javascript:", "tel:")

KEYWORDS = [
    "matrícula", "matricula", "horario", "cronograma", "calendario",
    "malla", "currícula", "curricula", "reglamento", "beca",
    "financiamiento", "pago", "pensión", "pension", "requisito",
    "ciclo", "crédito", "credito", "egreso", "ingresante",
    "grado", "titulación", "titulacion",
]

# Textos de botones que suelen abrir PDFs en SPAs
DOWNLOAD_BUTTON_TEXTS = [
    "ver malla", "descargar", "ver reglamento", "ver documento",
    "ver plan de estudios", "plan curricular", "ver pdf",
]

# ── Carpetas de salida (en /tmp para evitar iCloud) ───────────────────────────
OUTPUT_BASE = Path("/tmp/upc_documents")
SECTIONS    = ["pregrado", "becas", "matricula", "reglamentos", "otros"]
PDF_DIRS    = {s: OUTPUT_BASE / "pdfs"      / s for s in SECTIONS}
TXT_DIRS    = {s: OUTPUT_BASE / "html_text" / s for s in SECTIONS}
LOG_PATH    = OUTPUT_BASE / "log_scraping.csv"

for d in list(PDF_DIRS.values()) + list(TXT_DIRS.values()):
    d.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

print(f"Semillas       : {len(SEED_URLS)}")
print(f"Profundidad    : {MAX_DEPTH}")
print(f"SPA threshold  : {SPA_THRESHOLD} palabras")
print(f"Output base    : {OUTPUT_BASE}")

Semillas       : 15
Profundidad    : 2
SPA threshold  : 200 palabras
Output base    : /tmp/upc_documents


## 3 · Funciones auxiliares

In [18]:
def normalize_url(url: str) -> str:
    url, _ = urldefrag(url.strip())
    return url

def is_allowed_domain(url: str) -> bool:
    host = urlparse(url).hostname or ""
    return host == "upc.edu.pe" or host.endswith(ALLOWED_DOMAIN_SUFFIX)

def should_skip(url: str) -> str | None:
    if url.startswith(SKIP_SCHEMES):
        return "non-http-scheme"
    low = url.lower()
    for pat in SKIP_PATTERNS:
        if pat in low:
            return pat
    return None

def classify_section(url: str) -> str:
    host = urlparse(url).hostname or ""
    path = urlparse(url).path.lower()
    if host.startswith("pregrado."):
        return "pregrado"
    if "beca" in path or "financiamiento" in path or "cobranza" in path:
        return "becas"
    if "matricula" in path or "matrícula" in path:
        return "matricula"
    if "reglamento" in path:
        return "reglamentos"
    return "otros"

def extract_visible_text(soup: BeautifulSoup) -> str:
    work = BeautifulSoup(str(soup), "lxml")
    for tag in work(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
        tag.decompose()
    for sel in [".sidebar", ".menu", ".breadcrumb", "#wpadminbar"]:
        for el in work.select(sel):
            el.decompose()
    return re.sub(r"\s+", " ", work.get_text(separator=" ", strip=True))

def count_keywords(text: str) -> tuple[int, list[str]]:
    low = text.lower()
    hits = [kw for kw in KEYWORDS if kw in low]
    return len(hits), hits

def extract_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    seen, out = set(), []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href:
            continue
        absolute = normalize_url(urljoin(base_url, href))
        if absolute and absolute not in seen:
            seen.add(absolute)
            out.append(absolute)
    return out

def url_to_filename(url: str, ext: str) -> str:
    """Nombre de archivo único desde URL."""
    parsed = urlparse(url)
    slug = re.sub(r"[^a-z0-9_-]", "_", (parsed.hostname + parsed.path).lower())[:80]
    h = hashlib.md5(url.encode()).hexdigest()[:8]
    return f"{slug}_{h}{ext}"

# ── Smoke test ────────────────────────────────────────────────────────────────
assert classify_section("https://pregrado.upc.edu.pe/foo") == "pregrado"
assert classify_section("https://www.upc.edu.pe/admision/becas-y-financiamiento/") == "becas"
assert classify_section("https://www.upc.edu.pe/reglamento-academico") == "reglamentos"
assert should_skip("https://www.upc.edu.pe/login") == "login"
assert should_skip("mailto:info@upc.edu.pe") == "non-http-scheme"
print("helpers ✓")

helpers ✓


## 4 · Log CSV incremental

In [19]:
LOG_COLS = [
    "fecha_hora", "url", "profundidad", "tipo",
    "archivo_guardado", "palabras_extraidas", "notas",
]

class IncrementalLog:
    """Escribe cada evento al CSV en el momento que ocurre. Nada se pierde si el kernel muere."""
    def __init__(self, path: Path):
        self.file   = open(path, "w", newline="", encoding="utf-8")
        self.writer = csv.DictWriter(self.file, fieldnames=LOG_COLS)
        self.writer.writeheader()
        self.file.flush()

    def write(self, url, depth, tipo, archivo="", palabras=0, notas=""):
        self.writer.writerow({
            "fecha_hora"       : datetime.now().isoformat(timespec="seconds"),
            "url"              : url,
            "profundidad"      : depth,
            "tipo"             : tipo,
            "archivo_guardado" : archivo,
            "palabras_extraidas": palabras,
            "notas"            : notas,
        })
        self.file.flush()

    def close(self):
        self.file.close()

print("IncrementalLog ✓")

IncrementalLog ✓


## 5 · `fetch_simple(url)` — requests estático

Rápido y sin overhead. Retorna `ok=False` si el servidor responde error o si la URL no es HTML.

In [20]:
def fetch_simple(url: str) -> dict:
    result = {"ok": False, "status": None, "soup": None, "error": None, "method": "simple"}
    try:
        resp = session.get(url, timeout=TIMEOUT, allow_redirects=True)
        result["status"] = resp.status_code
        if resp.status_code != 200:
            result["error"] = f"HTTP {resp.status_code}"
            return result
        ctype = resp.headers.get("Content-Type", "").lower()
        if "html" not in ctype:
            result["error"] = f"non-html: {ctype[:50]}"
            return result
        result["soup"] = BeautifulSoup(resp.text, "lxml")
        result["ok"]   = True
    except requests.Timeout:
        result["error"] = "timeout"
    except requests.RequestException as e:
        result["error"] = f"request: {type(e).__name__}"
    except Exception as e:
        result["error"] = f"unexpected: {type(e).__name__}"
    return result

# Smoke test
_t = fetch_simple("https://pregrado.upc.edu.pe/facultad-de-ingenieria/")
text_preview = extract_visible_text(_t["soup"])[:120] if _t["ok"] else ""
print("fetch_simple:", "OK" if _t["ok"] else "FAIL", "|", _t["status"])
print("preview:", text_preview)

fetch_simple: OK | 200
preview: Facultad de IngenierÃ­a | Pregrado UPC FACULTAD DE INGENIERÃA Libera todo tu potencial. Transforma el mundo. Proyecta t


## 6 · `fetch_playwright(url)` — navegador headless

Usado cuando `fetch_simple` devuelve <`SPA_THRESHOLD` palabras.
- Abre Chromium headless y espera `networkidle` para que el JS hidrate el DOM
- Intercepta requests a `.pdf` lanzadas desde la página
- Intenta hacer click en botones de descarga para capturar PDFs dinámicos
- Reutiliza un browser singleton (no abre/cierra Chromium en cada URL)

In [21]:
import threading
from playwright.sync_api import TimeoutError as PWTimeout

def fetch_playwright(url: str) -> dict:
    """
    Playwright Sync API no puede correr dentro del event loop de Jupyter.
    Solución: abrir un thread OS separado — ahí no hay event loop previo.
    """
    _result = [{"ok": False, "status": None, "soup": None,
                "error": "thread-not-started", "method": "playwright",
                "new_pdf_urls": []}]

    def _run():
        # Import dentro del thread para evitar conflictos
        from playwright.sync_api import sync_playwright as _sync_pw, TimeoutError as _PWTimeout
        result = {"ok": False, "status": None, "soup": None,
                  "error": None, "method": "playwright", "new_pdf_urls": []}
        try:
            with _sync_pw() as pw:
                browser = pw.chromium.launch(headless=True)
                context = browser.new_context(
                    user_agent=USER_AGENT, ignore_https_errors=True
                )
                page      = context.new_page()
                pdf_found = []

                page.on("request", lambda req: pdf_found.append(req.url)
                        if req.url.lower().endswith(".pdf") else None)

                resp = page.goto(url, wait_until="domcontentloaded", timeout=30_000)
                result["status"] = resp.status if resp else None

                try:
                    page.wait_for_load_state("networkidle", timeout=12_000)
                except _PWTimeout:
                    pass  # continuar con lo que haya cargado

                for btn_text in DOWNLOAD_BUTTON_TEXTS:
                    try:
                        loc = page.locator(
                            f"button:has-text('{btn_text}'), a:has-text('{btn_text}')"
                        )
                        if loc.count() > 0:
                            loc.first.click(timeout=3_000)
                            page.wait_for_timeout(1_000)
                    except Exception:
                        pass

                html = page.content()
                context.close()
                browser.close()

                result["soup"]         = BeautifulSoup(html, "lxml")
                result["ok"]           = True
                result["new_pdf_urls"] = list(set(pdf_found))

        except Exception as e:
            result["error"] = f"playwright: {type(e).__name__}: {str(e)[:200]}"

        _result[0] = result

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout=50)  # 50s máximo por página
    if t.is_alive():
        _result[0]["error"] = "playwright-thread-timeout-50s"
    return _result[0]

def close_browser():
    pass  # con threading el browser cierra solo al salir de `with _sync_pw()`

# Smoke test
print("Iniciando Chromium en thread separado...")
_pw_test  = fetch_playwright("https://pregrado.upc.edu.pe/facultad-de-ingenieria/")
_pw_words = len(extract_visible_text(_pw_test["soup"]).split()) if _pw_test["ok"] else 0
print("status  :", _pw_test["status"])
print("ok      :", _pw_test["ok"])
print("error   :", _pw_test["error"])
print("palabras:", _pw_words)
print("PDFs    :", len(_pw_test.get("new_pdf_urls", [])))

Iniciando Chromium en thread separado...
status  : 200
ok      : True
error   : None
palabras: 1027
PDFs    : 0


## 7 · `fetch_smart(url)` — orquestador simple → Playwright

Intenta `fetch_simple`. Si hay menos de `SPA_THRESHOLD` palabras, cae a `fetch_playwright`. Si Playwright también falla, devuelve lo que tenía de simple para no perder la URL.

In [22]:
def fetch_smart(url: str) -> dict:
    result    = fetch_simple(url)
    if not result["ok"]:
        return result

    num_words = len(extract_visible_text(result["soup"]).split())

    if num_words >= SPA_THRESHOLD:
        return result  # estático con suficiente contenido, listo

    # Pocos palabras → SPA → intentar Playwright
    pw = fetch_playwright(url)
    if pw["ok"]:
        return pw

    # Playwright también falló → devolver resultado simple con nota
    result["notas"] = f"playwright-failed: {pw['error']}"
    return result

# Comparar simple vs smart en una URL de becas
TEST_URL = "https://www.upc.edu.pe/admision/becas-y-financiamiento/"

_simple = fetch_simple(TEST_URL)
_smart  = fetch_smart(TEST_URL)

w_simple = len(extract_visible_text(_simple["soup"]).split()) if _simple["ok"] else 0
w_smart  = len(extract_visible_text(_smart["soup"]).split())  if _smart["ok"]  else 0

print(f"fetch_simple  → {w_simple:>5d} palabras")
print(f"fetch_smart   → {w_smart:>5d} palabras  (method={_smart.get('method','?')})")

if w_smart > w_simple:
    print(f"\n✓ Playwright sumó {w_smart - w_simple} palabras extra")
else:
    print("\n✓ simple ya tenía suficiente contenido, Playwright no fue necesario")

fetch_simple  →   355 palabras
fetch_smart   →   355 palabras  (method=simple)

✓ simple ya tenía suficiente contenido, Playwright no fue necesario


## 8 · `download_pdf(url, section, depth, log)` — descarga PDFs

Guarda el PDF en la subcarpeta correcta. Si el archivo ya existe lo salta (idempotente). Si la URL redirigió a un HTML (página de login enmascarada) lo registra y descarta.

In [23]:
def download_pdf(url: str, section: str, depth: int, log: IncrementalLog) -> str | None:
    dest = PDF_DIRS[section] / url_to_filename(url, ".pdf")

    if dest.exists():
        log.write(url, depth, "pdf", str(dest), notas="ya-existia-skip")
        return str(dest)

    try:
        resp = session.get(url, timeout=30, stream=True, allow_redirects=True)
        if resp.status_code != 200:
            log.write(url, depth, "error", notas=f"HTTP {resp.status_code}")
            return None
        # Algunas URLs .pdf redirigen silenciosamente a HTML (login, etc.)
        ctype = resp.headers.get("Content-Type", "").lower()
        if "html" in ctype:
            log.write(url, depth, "ignorada", notas="pdf-url-redirigió-a-html")
            return None
        with open(dest, "wb") as f:
            shutil.copyfileobj(resp.raw, f)
        size_kb = dest.stat().st_size / 1024
        log.write(url, depth, "pdf", str(dest), notas=f"{size_kb:.0f}KB")
        return str(dest)
    except Exception as e:
        log.write(url, depth, "error", notas=f"pdf-download: {type(e).__name__}")
        return None

# Smoke test — descargar un PDF real de pregrado
print("Descargando PDF de prueba...")
_test_log = IncrementalLog(OUTPUT_BASE / "test_log.csv")
_test_pdf = download_pdf(
    "https://pregrado.upc.edu.pe/wp-content/uploads/2023/07/Malla-ISI-2023-1.pdf",
    "pregrado", 1, _test_log
)
_test_log.close()
if _test_pdf:
    size = Path(_test_pdf).stat().st_size / 1024
    print(f"✓ descargado: {Path(_test_pdf).name}  ({size:.0f} KB)")
else:
    print("✗ falló — URL puede haber cambiado, el scraper real la encontrará dinámicamente")

Descargando PDF de prueba...
✗ falló — URL puede haber cambiado, el scraper real la encontrará dinámicamente


## 9 · `process_page(url, depth, log, pdf_queue, link_queue, visited)` — lógica de una URL

Orquesta fetch → extrae texto → guarda `.txt` si es relevante → encola links hijos y PDFs encontrados.

In [24]:
def process_page(
    url: str,
    depth: int,
    log: IncrementalLog,
    pdf_queue: list,
    link_queue: deque,
    visited: set,
):
    section = classify_section(url)
    result  = fetch_smart(url)

    if not result["ok"]:
        log.write(url, depth, "error", notas=result.get("error", ""))
        return

    soup      = result["soup"]
    method    = result.get("method", "simple")
    text      = extract_visible_text(soup)
    num_words = len(text.split())
    num_kw, kw_hits = count_keywords(text)

    # PDFs interceptados por Playwright durante el render
    for pdf_url in result.get("new_pdf_urls", []):
        norm = normalize_url(pdf_url)
        if norm not in visited:
            visited.add(norm)
            pdf_queue.append((norm, classify_section(norm), depth + 1))

    # Extraer y clasificar todos los links de la página
    for link in extract_links(soup, url):
        link = normalize_url(link)
        if link in visited:
            continue
        if should_skip(link) or not is_allowed_domain(link):
            continue
        if link.lower().endswith(".pdf"):
            visited.add(link)
            pdf_queue.append((link, classify_section(link), depth + 1))
        elif depth < MAX_DEPTH:
            link_queue.append((link, depth + 1))

    # ¿Vale la pena guardar el texto?
    relevant = num_words >= 300 and num_kw >= 1
    if relevant:
        title_tag  = soup.find("title")
        title      = title_tag.string.strip() if title_tag and title_tag.string else urlparse(url).path
        dest       = TXT_DIRS[section] / url_to_filename(url, ".txt")
        header     = (
            f"URL: {url}\n"
            f"Título: {title}\n"
            f"Sección: {section}\n"
            f"Fecha: {datetime.now().date()}\n"
            f"Keywords: {', '.join(kw_hits)}\n\n"
        )
        dest.write_text(header + text, encoding="utf-8")
        log.write(url, depth, "html", str(dest), num_words,
                  notas=f"method={method},kw={num_kw}")
    else:
        log.write(url, depth, "ignorada", notas=f"words={num_words},kw={num_kw},method={method}")

# Smoke test — procesar una URL y ver qué se guarda
print("Probando process_page en explora.upc.edu.pe/ayuda-upc ...")
_plog   = IncrementalLog(OUTPUT_BASE / "test_process.csv")
_pqueue = []
_lqueue = deque()
_vis    = set()
process_page("https://explora.upc.edu.pe/ayuda-upc", 1, _plog, _pqueue, _lqueue, _vis)
_plog.close()

import pandas as pd
_plog_df = pd.read_csv(OUTPUT_BASE / "test_process.csv")
print(_plog_df[["tipo","palabras_extraidas","notas"]].to_string(index=False))
print(f"\nPDFs encolados  : {len(_pqueue)}")
print(f"Links encolados : {len(_lqueue)}")

Probando process_page en explora.upc.edu.pe/ayuda-upc ...
tipo  palabras_extraidas               notas
html               13483 method=simple,kw=17

PDFs encolados  : 2
Links encolados : 53


## 10 · `scrape(seed_urls)` — BFS completo

BFS depth-limited. Por cada URL: `process_page` → descarga PDFs encolados → delay → siguiente.  
Escribe el log línea a línea. Si el kernel muere, todo lo ya procesado queda en disco.

In [25]:
def scrape(seed_urls: list[str]) -> None:
    log       = IncrementalLog(LOG_PATH)
    visited   = set()
    link_queue = deque()
    pdf_queue  = []

    for seed in seed_urls:
        link_queue.append((normalize_url(seed), 0))

    pages_done = 0
    pdfs_done  = 0
    txt_saved  = 0

    pbar = tqdm(total=MAX_PAGES, desc="scraping", unit="url")

    try:
        while link_queue and pages_done < MAX_PAGES:
            url, depth = link_queue.popleft()
            if url in visited:
                continue
            visited.add(url)

            skip = should_skip(url)
            if skip or not is_allowed_domain(url):
                log.write(url, depth, "ignorada", notas=skip or "external-domain")
                continue

            process_page(url, depth, log, pdf_queue, link_queue, visited)
            pages_done += 1

            # Contar TXTs guardados leyendo el log en memoria es caro;
            # aproximamos contando archivos en disco cada 20 páginas
            if pages_done % 20 == 0:
                txt_saved = sum(len(list(d.glob("*.txt"))) for d in TXT_DIRS.values())

            pbar.update(1)
            pbar.set_postfix(pdfs=pdfs_done, txt=txt_saved, q=len(link_queue))

            # Descargar todos los PDFs encolados antes de la siguiente página
            while pdf_queue:
                pdf_url, pdf_section, pdf_depth = pdf_queue.pop(0)
                result = download_pdf(pdf_url, pdf_section, pdf_depth, log)
                if result:
                    pdfs_done += 1
                pbar.set_postfix(pdfs=pdfs_done, txt=txt_saved, q=len(link_queue))

            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    finally:
        pbar.close()
        log.close()
        close_browser()
        # Conteo final real desde disco
        total_txt  = sum(len(list(d.glob("*.txt"))) for d in TXT_DIRS.values())
        total_pdfs = sum(len(list(d.glob("*.pdf"))) for d in PDF_DIRS.values())
        print(f"\n{'='*50}")
        print(f"Páginas procesadas : {pages_done}")
        print(f"TXTs guardados     : {total_txt}")
        print(f"PDFs descargados   : {total_pdfs}")
        print(f"Log                : {LOG_PATH}")
        print(f"{'='*50}")

print("scrape() listo ✓")

scrape() listo ✓


## 11 · Ejecución

Con 15 semillas, `MAX_PAGES=600`, delays 2-4s y Playwright para SPAs, esperar **50-70 min**.  
Los archivos van apareciendo en `/tmp/upc_documents/` en tiempo real.  
Puedes monitorear desde terminal: `watch -n 10 'find /tmp/upc_documents -name "*.pdf" | wc -l'`

In [26]:
started_at = datetime.now()
scrape(SEED_URLS)
elapsed = (datetime.now() - started_at).total_seconds()
print(f"Duración total: {elapsed:.0f}s  ({elapsed/60:.1f} min)")

scraping:   0%|          | 0/600 [00:00<?, ?url/s]


Páginas procesadas : 600
TXTs guardados     : 237
PDFs descargados   : 132
Log                : /tmp/upc_documents/log_scraping.csv
Duración total: 3588s  (59.8 min)


## 12 · Resumen de resultados

In [27]:
import pandas as pd

log_df = pd.read_csv(LOG_PATH)

print("=== Resumen por tipo ===")
print(log_df["tipo"].value_counts())

print("\n=== PDFs por sección ===")
pdfs = log_df[log_df["tipo"] == "pdf"].copy()
pdfs["seccion"] = pdfs["archivo_guardado"].apply(
    lambda p: Path(p).parent.name if pd.notna(p) and p else "?"
)
print(pdfs.groupby("seccion").size().rename("cantidad"))

print("\n=== TXTs por sección ===")
html_df = log_df[log_df["tipo"] == "html"].copy()
html_df["seccion"] = html_df["archivo_guardado"].apply(
    lambda p: Path(p).parent.name if pd.notna(p) and p else "?"
)
print(html_df.groupby("seccion").agg(
    paginas=("url", "count"),
    palabras_total=("palabras_extraidas", "sum"),
    palabras_prom=("palabras_extraidas", "mean"),
).round(0))

print("\n=== Errores más comunes ===")
errors = log_df[log_df["tipo"] == "error"]
if not errors.empty:
    print(errors["notas"].value_counts().head(10))
else:
    print("Sin errores")

print("\n=== Archivos en disco ===")
for s in SECTIONS:
    n_pdf = len(list(PDF_DIRS[s].glob("*.pdf")))
    n_txt = len(list(TXT_DIRS[s].glob("*.txt")))
    if n_pdf or n_txt:
        print(f"  {s:12s}  {n_pdf:3d} PDFs   {n_txt:3d} TXTs")

=== Resumen por tipo ===
tipo
ignorada    348
html        237
pdf         132
error        32
Name: count, dtype: int64

=== PDFs por sección ===
seccion
becas         11
matricula      1
otros        118
pregrado       2
Name: cantidad, dtype: int64

=== TXTs por sección ===
             paginas  palabras_total  palabras_prom
seccion                                            
becas             26           25419          978.0
matricula          2            2601         1300.0
otros            192          294140         1532.0
pregrado          15           13174          878.0
reglamentos        2            1582          791.0

=== Errores más comunes ===
notas
HTTP 404                        25
HTTP 403                         2
HTTP 500                         1
non-html: application/pdf        1
non-html: application/msword     1
request: ConnectionError         1
request: InvalidSchema           1
Name: count, dtype: int64

=== Archivos en disco ===
  pregrado        2 PDFs  